# Política de Decisão

Após a comparação dos algoritmos de aprendizado de máquina,
o CatBoost foi selecionado como modelo preditivo para as etapas
seguintes do projeto.

O modelo produz, para cada transação, um score contínuo associado
ao risco de fraude. Entretanto, esse score, isoladamente, ainda não
representa uma decisão operacional.

O objetivo desta etapa é transformar o score produzido pelo modelo
em três possíveis ações:

- **APROVAR**: transações com baixo risco estimado;
- **REVISAR**: transações com risco intermediário, encaminhadas para
  análise;
- **ALERTA_CRÍTICO**: transações com risco elevado, tratadas com maior
  prioridade.

A definição dos pontos de corte será realizada exclusivamente sobre
o conjunto de validação. O conjunto de teste permanecerá intocado
até que o modelo e a política de decisão estejam completamente
congelados.

In [44]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [45]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(
    0,
    str(PROJECT_ROOT)
)

print(
    "Project root:",
    PROJECT_ROOT
)

Project root: /Users/lucassantos/Documents/ccard_fraud_ml


In [46]:
from src.inference import FraudModel

In [47]:
modelo_fraude = FraudModel()

In [48]:
print(
    "Árvores carregadas:",
    modelo_fraude.modelo.tree_count_
)

Árvores carregadas: 719


In [49]:
DATA_PATH = (
    PROJECT_ROOT
    / "raw"
    / "fraudTrain.csv"
)

In [50]:
dtypes = {
    "cc_num": "string",
    "trans_num": "string",
    "zip": "string",
    "merchant": "string",
    "category": "category",
    "gender": "category",
    "state": "category",
    "first": "string",
    "last": "string",
    "street": "string",
    "city": "string",
    "job": "string"
}

In [51]:
df = pd.read_csv(
    DATA_PATH,
    dtype=dtypes,
    parse_dates=[
        "trans_date_trans_time",
        "dob"
    ]
)

In [52]:
colunas_indice = [
    coluna
    for coluna in df.columns
    if str(coluna).startswith(
        "Unnamed:"
    )
]

if colunas_indice:
    df = df.drop(
        columns=colunas_indice
    )

In [53]:
print(
    "Transações:",
    f"{len(df):,}"
)

print(
    "Fraudes:",
    f"{df['is_fraud'].sum():,}"
)

Transações: 1,296,675
Fraudes: 7,506


In [54]:
df = (
    df
    .sort_values(
        "trans_date_trans_time"
    )
    .reset_index(drop=True)
)

In [55]:
print(
    "Primeira transação:",
    df[
        "trans_date_trans_time"
    ].min()
)

print(
    "Última transação:",
    df[
        "trans_date_trans_time"
    ].max()
)

Primeira transação: 2019-01-01 00:00:18
Última transação: 2020-06-21 12:13:37


In [56]:
n_total = len(df)

fim_treino = int(
    n_total * 0.70
)

fim_validacao = int(
    n_total * 0.85
)

In [57]:
df_treino = (
    df.iloc[
        :fim_treino
    ]
    .copy()
)

df_validacao = (
    df.iloc[
        fim_treino:fim_validacao
    ]
    .copy()
)

df_teste = (
    df.iloc[
        fim_validacao:
    ]
    .copy()
)

In [58]:
print(
    "Treino:",
    f"{len(df_treino):,}"
)

print(
    "Validação:",
    f"{len(df_validacao):,}"
)

print(
    "Teste:",
    f"{len(df_teste):,}"
)

Treino: 907,672
Validação: 194,501
Teste: 194,502


In [59]:
print(
    "Fraudes na validação:",
    f"{df_validacao['is_fraud'].sum():,}"
)

print(
    "Legítimas na validação:",
    f"{(df_validacao['is_fraud'] == 0).sum():,}"
)

Fraudes na validação: 1,252
Legítimas na validação: 193,249


In [60]:
assert len(df_validacao) == 194_501

assert (
    df_validacao[
        "is_fraud"
    ].sum()
    == 1_252
)

print(
    "Split de validação: OK"
)

Split de validação: OK


In [61]:
y_validacao = (
    df_validacao[
        "is_fraud"
    ]
    .to_numpy()
)

In [62]:
scores_validacao = (
    modelo_fraude.prever_scores(
        df_validacao
    )
)

TypeError: Cannot setitem on a Categorical with a new category (__MISSING__), set the categories first

In [63]:
import src.features as features

print(features.__file__)

/Users/lucassantos/Documents/ccard_fraud_ml/src/features.py


In [64]:
import inspect

print(
    inspect.getsource(
        features.construir_features
    )
)

def construir_features(
    df: pd.DataFrame
) -> pd.DataFrame:

    dados = df.copy()

    dados[
        "trans_date_trans_time"
    ] = pd.to_datetime(
        dados[
            "trans_date_trans_time"
        ]
    )

    dados["dob"] = pd.to_datetime(
        dados["dob"]
    )

    # Valor
    dados["amt_log"] = np.log1p(
        dados["amt"]
    )

    # População
    dados["city_pop_log"] = np.log1p(
        dados["city_pop"]
    )

    # Idade
    transacao = (
        dados[
            "trans_date_trans_time"
        ]
    )

    nascimento = dados["dob"]

    dados["idade"] = (
        transacao.dt.year
        -
        nascimento.dt.year
        -
        (
            (
                transacao.dt.month
                <
                nascimento.dt.month
            )
            |
            (
                (
                    transacao.dt.month
                    ==
                    nascimento.dt.month
                )
                &
                (
